# Notebook 03: Estimation, bivariate analysis and regression

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 3

NexusHealth · Adamu Onu

You will choose methods that match the question, compare groups with effect sizes and confidence intervals, and fit linear and logistic regression models to estimate the association between medication adherence and blood pressure, adjusting for confounders.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
})

# Find the course folder, wherever this notebook lives inside it.
ROOT <- normalizePath(getwd())
while (!dir.exists(file.path(ROOT, "data", "raw")) && dirname(ROOT) != ROOT) {
  ROOT <- dirname(ROOT)
}
stopifnot("Course folder not found. Open the whole course folder in VS Code." =
            dir.exists(file.path(ROOT, "data", "raw")))

DATA_RAW       <- file.path(ROOT, "data", "raw")
DATA_PROCESSED <- file.path(ROOT, "data", "processed")
DATA_REFERENCE <- file.path(ROOT, "data", "reference")
OUT_TABLES     <- file.path(ROOT, "outputs", "tables")
OUT_FIGURES    <- file.path(ROOT, "outputs", "figures")
for (folder in c(DATA_PROCESSED, OUT_TABLES, OUT_FIGURES)) dir.create(folder, recursive = TRUE, showWarnings = FALSE)

options(repr.plot.width = 7, repr.plot.height = 4.5, pillar.width = 120)
theme_set(theme_minimal(base_size = 12))
cat("Course folder found:", basename(ROOT), "\n")

In [ ]:
analysis_path <- file.path(DATA_PROCESSED, "analysis_set.csv")
if (!file.exists(analysis_path)) {
  analysis_path <- file.path(DATA_REFERENCE, "analysis_set_reference.csv")
  message("Your Day 2 file was not found, so the reference analysis set is used instead.")
}
df <- read_csv(analysis_path, show_col_types = FALSE) |>
  mutate(
    # Tell R which variables are categories, and which level is the reference.
    sex            = factor(sex, levels = c("Female", "Male")),
    smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current")),
    clinic         = factor(clinic, levels = c("North", "East", "South", "West"))
  )
cat("Loaded", basename(analysis_path), ":", nrow(df), "rows,", ncol(df), "columns\n")

In [ ]:
library(broom)
BRAND <- "#002366"

# Part A: Estimation and method selection (block 1)

Before choosing a test, write down the **estimand**: the quantity you want to know.

> *Among adults in primary care, how much does mean systolic BP differ per 10-point higher adherence score, among patients of the same age, sex, BMI, diabetes status and smoking status?*

Then the method follows from the type of outcome and exposure:

| Outcome | Exposure | Bivariate method | Effect measure | Multivariable model |
|---|---|---|---|---|
| Continuous (SBP) | Two groups | Welch t-test | Difference in means | Linear regression |
| Continuous | Continuous | Correlation | Pearson r (or Spearman) | Linear regression |
| Binary (uncontrolled BP) | Categories | Chi-square test | Difference in proportions | Logistic regression |
| Binary | Continuous | (go straight to a model) | Odds ratio | Logistic regression |

**A confidence interval** gives the range of values for the true effect that are reasonably compatible with the data. **A p-value** tells you how surprising the data would be if there were truly no effect. Neither tells you whether the effect matters clinically; that is your call.

## A1. A mean with a 95% confidence interval

The interval is the estimate plus or minus about two standard errors.

In [ ]:
sbp <- df$systolic_bp
m <- mean(sbp)
se <- sd(sbp) / sqrt(length(sbp))
t_crit <- qt(0.975, df = length(sbp) - 1)
cat(sprintf("Mean SBP %.1f mmHg, 95%% CI %.1f to %.1f (n = %d)\n", m, m - t_crit * se, m + t_crit * se, length(sbp)))

## Exercise 3.0: Choose the method

For each question, name the outcome type, the exposure type, a suitable bivariate method and the effect measure you would report.

1. Is mean BMI different between current smokers and never smokers?
2. Is the proportion with uncontrolled BP different between clinics?
3. Is higher adherence associated with uncontrolled BP after allowing for age and diabetes?

**Your answer:** _Double-click here and write your answer._

# Part B: Bivariate analysis (block 2)

## B1. Comparing two means: SBP by sex

Always start with a description and a picture, then the test.

In [ ]:
df |>
  group_by(sex) |>
  summarise(n = n(), mean = mean(systolic_bp), sd = sd(systolic_bp), median = median(systolic_bp)) |>
  mutate(across(where(is.double), ~ round(.x, 1)))

ggplot(df, aes(sex, systolic_bp)) +
  geom_boxplot(fill = "#DEE6EF") +
  labs(x = NULL, y = "SBP (mmHg)", title = "Systolic BP by sex")

We use **Welch's t-test**, which does not assume the two groups have equal variances. It is a safe default. We report the **difference in means with its 95% CI**, not just the p-value.

In [ ]:
male <- df$systolic_bp[df$sex == "Male"]
female <- df$systolic_bp[df$sex == "Female"]
welch <- t.test(male, female, var.equal = FALSE)
cat(sprintf("Difference (male minus female): %.1f mmHg, 95%% CI %.1f to %.1f, p = %.3f\n",
            mean(male) - mean(female), welch$conf.int[1], welch$conf.int[2], welch$p.value))

## Exercise 3A (part 1): SBP by diabetes

Compare systolic BP between patients with and without diabetes. Give group sizes and means, a boxplot, and the difference (diabetes minus no diabetes) with 95% CI and p-value.

**AI assist.** Ask Copilot to *explain* the Welch test output line by line. Then check which group was subtracted from which. Getting the direction wrong is one of the most common errors.

In [ ]:
# Group summaries
...

# Boxplot
...

# Welch t-test: diabetes minus no diabetes
...

**Interpret in one or two sentences, and give one limitation.**

**Your answer:** _Double-click here and write your answer._

## B2. Comparing proportions: uncontrolled BP by smoking status

For two categorical variables, start with a cross-table showing **row percentages**, then use a chi-square test. Check the *expected* counts: the test is unreliable if many are below 5.

In [ ]:
table <- table(df$smoking_status, df$uncontrolled_bp)
print(table)
print(round(100 * prop.table(table, margin = 1), 1))

chi <- chisq.test(table)
cat(sprintf("Chi-square = %.2f, df = %d, p = %.3f; smallest expected count = %.1f\n",
            chi$statistic, chi$parameter, chi$p.value, min(chi$expected)))

## Exercise 3A (part 2): Uncontrolled BP by diabetes

Make the cross-table with row percentages, run the chi-square test, and report the **difference in proportions** (diabetes minus no diabetes) in percentage points.

Note: for a 2 x 2 table both Python and R apply a small *continuity correction* by default.

In [ ]:
# Cross-table and row percentages
...

# Chi-square test
...

# Difference in proportions (percentage points)
...

## B3. Correlation: adherence and SBP

Plot first. A correlation coefficient summarises a *straight-line* relationship in one number between -1 and 1. Spearman's version uses ranks and is less affected by outliers.

In [ ]:
ggplot(df, aes(medication_adherence, systolic_bp)) +
  geom_point(alpha = 0.4, colour = BRAND) +
  geom_smooth(method = "lm", formula = y ~ x, colour = "#A2B5CD") +
  labs(x = "Adherence score (%)", y = "SBP (mmHg)", title = "Adherence and systolic BP")

pearson <- cor.test(df$medication_adherence, df$systolic_bp, method = "pearson")
spearman <- cor.test(df$medication_adherence, df$systolic_bp, method = "spearman", exact = FALSE)
cat(sprintf("Pearson r = %.2f (p = %.3f); Spearman rho = %.2f (p = %.3f)\n",
            pearson$estimate, pearson$p.value, spearman$estimate, spearman$p.value))

A correlation of about -0.13 is weak: adherence explains under 2% of the variation in SBP. It can still matter, because a small shift in the average BP of a whole population can be clinically important. Correlation also cannot adjust for confounders, which is why we now turn to regression.

# Part C: Linear regression (block 3)

Linear regression estimates how the **mean** of a continuous outcome changes with each predictor, holding the other predictors constant.

`SBP = intercept + b1 x adherence_10 + b2 x age + ... + error`

* `b1` is the difference in mean SBP for a 10-point higher adherence score, among people with the same values of the other variables.
* For categories (sex, smoking), each coefficient compares a level with the **reference level** (Female, Never).

Which covariates? Those on our Day 1 framework map as plausible **confounders** (they affect both adherence and BP). We do not adjust for mediators or colliders.

## C1. The unadjusted (crude) model

In [ ]:
crude_lm <- lm(systolic_bp ~ adherence_10, data = df)
tidy(crude_lm, conf.int = TRUE)

**Reading the output.** The intercept is the predicted SBP at an adherence of 0 (rarely useful by itself). The `adherence_10` coefficient is the one that answers our question.

## Exercise 3B: Fit, check and interpret an adjusted linear model

1. Fit the adjusted model with `adherence_10`, `age`, `sex`, `bmi`, `diabetes` and `smoking_status`. Wrap diabetes in `factor()` so R treats it as a category.
2. Make a small table comparing the adherence estimate and 95% CI in the crude and adjusted models.

In [ ]:
adjusted_lm <- lm(systolic_bp ~ ..., data = df)
tidy(adjusted_lm, conf.int = TRUE)

# Compare the adherence estimate in the two models
...

**Interpret the adjusted estimate** in one sentence a colleague could read, then say why it changed from the crude estimate.

**Your answer:** _Double-click here and write your answer._

## C2. Check the model's assumptions

Linear regression assumes the residuals (observed minus predicted) are roughly symmetric, have similar spread across predicted values, and have no strong pattern. Two plots cover most of this.

In [ ]:
diag <- augment(adjusted_lm)
p_resid <- ggplot(diag, aes(.fitted, .resid)) +
  geom_point(alpha = 0.4, colour = BRAND) +
  geom_hline(yintercept = 0, linetype = "dashed", colour = "grey40") +
  labs(x = "Fitted SBP (mmHg)", y = "Residual (mmHg)", title = "Residuals vs fitted")
p_qq <- ggplot(diag, aes(sample = .resid)) +
  stat_qq(alpha = 0.4, colour = BRAND) + stat_qq_line() +
  labs(title = "Normal Q-Q plot of residuals", x = "Theoretical quantiles", y = "Sample quantiles")
print(p_resid); print(p_qq)
ggsave(file.path(OUT_FIGURES, "fig_linear_diagnostics_resid.png"), p_resid, width = 5, height = 4, dpi = 300)
cat(sprintf("R-squared: %.3f\n", summary(adjusted_lm)$r.squared))

**Diagnostic note.** In two sentences, say what the plots show and whether it changes your conclusion.

**Your answer:** _Double-click here and write your answer._

# Part D: Logistic regression (block 4)

For a binary outcome (uncontrolled BP yes/no), logistic regression models the **log-odds** of the outcome. Exponentiating a coefficient gives an **odds ratio (OR)**.

| Scale | Meaning | Example |
|---|---|---|
| Probability (risk) | Share of people with the outcome | 0.69 = 69% |
| Odds | Probability / (1 - probability) | 0.69 / 0.31 = 2.2 |
| Odds ratio | Odds in one group / odds in another | OR 0.8 = odds 20% lower |

> **Odds are not risks.** When the outcome is common (here about 69%), an OR of 0.8 does **not** mean a 20% lower risk; the change in risk is smaller. Use predicted probabilities to talk about risk.

In [ ]:
crude_logit <- glm(uncontrolled_bp ~ adherence_10, data = df, family = binomial)
tidy(crude_logit, conf.int = TRUE, exponentiate = TRUE)

## Exercise 3C: Write a defensible logistic interpretation

1. Fit the adjusted logistic model with the same covariates as the linear model.
2. Show odds ratios with 95% CIs. (R's `tidy()` gives profile-likelihood intervals; Python gives Wald intervals. They can differ slightly in the third decimal place.)
3. Calculate the average predicted probability of uncontrolled BP if everyone had an adherence score of 40, 60 and 80 (keeping their other characteristics as they are).

In [ ]:
# 1. Adjusted model
adjusted_logit <- ...

# 2. Odds ratios with 95% CI
...

# 3. Average predicted probabilities at adherence 40, 60 and 80
...

**Write three sentences:** the adjusted OR with its CI, what it means in words, and what the predicted probabilities add.

**Your answer:** _Double-click here and write your answer._

## D2. Save the model results for Day 4

In [ ]:
results <- bind_rows(
  tidy(crude_lm, conf.int = TRUE) |> mutate(model = "Linear, crude", n = nobs(crude_lm)),
  tidy(adjusted_lm, conf.int = TRUE) |> mutate(model = "Linear, adjusted", n = nobs(adjusted_lm)),
  tidy(crude_logit, conf.int = TRUE, exponentiate = TRUE) |> mutate(model = "Logistic, crude", n = nobs(crude_logit)),
  tidy(adjusted_logit, conf.int = TRUE, exponentiate = TRUE) |> mutate(model = "Logistic, adjusted", n = nobs(adjusted_logit))
) |>
  select(model, term, estimate, ci_low = conf.low, ci_high = conf.high, p_value = p.value, n)
write_csv(results, file.path(OUT_TABLES, "model_results.csv"))
results |> filter(term == "adherence_10") |> mutate(across(where(is.double), ~ round(.x, 3)))

## Key points

* State the estimand first; the method follows from the outcome and exposure types.
* Report effect sizes with 95% CIs. A p-value alone says nothing about size or clinical importance.
* Check the direction of every comparison (which group minus which).
* Adjust for confounders chosen from your framework, not for every variable you have.
* Odds ratios are not risk ratios when the outcome is common. Use predicted probabilities for risk.
* Running many tests and reporting only the significant ones is misleading. Plan your analyses in advance.